# 07 · Fase 2: efectos entre categorías

Se estima, para cada una de las **10 categorías del núcleo** (D‑28, D‑30), la ecuación de la fase 2 de `metodologia.tex`: la de la fase 1 más la intensidad promocional de las otras 9 categorías del núcleo y el resto como control. Se reportan solo los efectos entre categorías $\beta_{jk}$; los efectos propios y dentro de la categoría están en la fase 1.

$$
\mathbb{E}[y_{it}] = \exp\Big(\alpha_i + \gamma_{\text{dow}(t)} + \tau_{m(t)} + \mathbf{h}_t'\boldsymbol{\theta}
+ \sum_{b} \beta_b D^{(b)}_{it} + \sum_{b} \psi^{\text{cl}}_b S^{\text{cl}(b)}_{it} + \sum_{b} \psi^{\text{cat}}_b S^{\text{cat}(b)}_{it}
+ \sum_{k\in\mathcal{K}\setminus\{j\}}\sum_{b} \beta_{jk,b} P^{(b)}_{kt} + \sum_{b} \rho_b R^{(b)}_{jt}\Big),
\qquad b \in \{0,\ 1\text{-}3,\ 4\text{-}6\}
$$

- PPML con efectos fijos de SKU, día de la semana y año‑mes; errores agrupados por SKU y semana (D‑08, D‑15, D‑16).
- MEATS, POULTRY, DELI y PRODUCE se estiman sin sus días de programa; sus resultados se refieren a promociones **fuera del programa** (D‑26).
- $D_{it}$, $S^{\text{cl}}_{it}$, $S^{\text{cat}}_{it}$ y $R_{jt}$ (`p__RESTO`) entran como parte de la especificación, pero aquí no se interpretan.

Salidas en `resultados/tablas/` (prefijo `fase2_`, incluido un Excel con todas las tablas). La matriz de efectos entre categorías (fase 4) se construye en `08_fase4_matriz` a partir de `fase2_efectos_cruzados.csv`.

## Paso 1. Librerías y parámetros

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook estima la fase 2 (efectos entre categorías) para las 10 categorías del núcleo.
# ------------------------------------------------------------------------------------------------------------
# Implementación: grupo de tesis
# Fecha: 2026-11-07
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import re
import warnings

import numpy as np
import pandas as pd
import pyfixest as pf
from scipy import stats
from statsmodels.stats.multitest import multipletests

# pyfixest avisa con este mensaje cuando un control de calendario de uno o dos días queda sin error estándar
warnings.filterwarnings("ignore", message="invalid value encountered in sqrt")

# Parámetros
# ------------------------------------------------------------------------------------------------------------
# D-28 y D-30: categorías objetivo = núcleo de exposición; primero las sin programa, luego las con programa (D-26)
categorias = [
    "BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER",
    "MEATS", "POULTRY", "DELI", "PRODUCE",
]
categorias_programa = ["MEATS", "POULTRY", "DELI", "PRODUCE"]

# D-14: día de la promoción y promedios de los días 1-3 y 4-6 anteriores (sin rezago 7 ni adelantos, D-29)
bloques = ["", "_l1_3", "_l4_6"]
nombre_bloque = {"": "dia", "_l1_3": "l1_3", "_l4_6": "l4_6"}

# Regresores de la fase 1: promoción propia, de los otros SKUs de la clase y del resto de la categoría
regresores_fase1 = [f"{v}{b}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for b in bloques]

# D-10 y D-13: controles de calendario (los mismos de la fase 1)
controles_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]

variable_dependiente = "unit_sales_ppml"   # D-08, D-20
efectos_fijos = "item_nbr + dow + mes"     # D-16
agrupamiento = "item_nbr + semana"         # D-15
delta_p = 0.10                             # efectos de proporciones: +10 puntos porcentuales del surtido
alfa = 0.05                                # D-15: Benjamini-Hochberg al 5 %

# Columna de intensidad de cada categoría del núcleo (mismo criterio de nombres que 03_exposiciones)
columna_categoria = {c: "p__" + re.sub(r"[^A-Z0-9]+", "_", c).strip("_") for c in categorias}

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_procesados = os.path.join("..", "data", "procesados")
ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
os.makedirs(ruta_tablas, exist_ok=True)
os.makedirs(ruta_figuras, exist_ok=True)

## Paso 2. Importar los datos

Los CSV se leen con `parse_dates` para que la unión con los Parquet no pierda filas.

In [2]:
panel = pd.read_parquet(os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet"))
exposiciones = pd.read_parquet(os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet"))
calendario = pd.read_csv(os.path.join(ruta_procesados, "calendario_t44.csv"), parse_dates=["date"])
exposicion_categorias = pd.read_csv(
    os.path.join(ruta_procesados, "exposicion_t44_categorias_dia.csv"), parse_dates=["date"]
)
exposicion_resto = pd.read_csv(
    os.path.join(ruta_procesados, "exposicion_t44_resto_por_objetivo.csv"), parse_dates=["date"]
)

# Cifras de control del README
assert len(panel) == len(exposiciones) == 1_935_758, "el panel o las exposiciones no tienen 1.935.758 filas"
assert len(calendario) == len(exposicion_categorias) == 590, "las tablas diarias no tienen 590 días"
assert set(columna_categoria.values()) <= set(exposicion_categorias.columns), "falta alguna categoría del núcleo"

## Paso 3. Armar la base de cada categoría

Sigue `INSTRUCCIONES.md` §6 y §7. Las variables de otras categorías varían solo por fecha, y sus nulos coinciden con los de los rezagos de la fase 1. Por eso el segundo `assert` exige que la muestra sea **la misma de la fase 1**, de modo que los efectos propios de ambas fases sean comparables.

In [3]:
def armar_base(categoria: str) -> tuple:
    """
    Esta función arma la base de estimación de la fase 2 para una categoría objetivo.

    :param categoria:   el nombre de la categoría (columna family).
    :return:            (base con una fila por SKU-día usada en la estimación, lista de regresores de promoción).
    """

    # Regresores: fase 1 + las otras 9 categorías del núcleo (sin la propia, D-28) + el resto como control
    cruzadas = [f"{columna_categoria[k]}{b}" for k in categorias if k != categoria for b in bloques]
    resto = [f"p__RESTO{b}" for b in bloques]
    regresores = regresores_fase1 + cruzadas + resto

    # Unir panel, exposiciones SKU-día, calendario, intensidad del núcleo y resto de la categoría objetivo
    base = panel.loc[panel["family"] == categoria]
    filas = len(base)
    base = base.merge(exposiciones, on=["item_nbr", "date"], how="left")
    base = base.merge(calendario, on="date", how="left")
    base = base.merge(exposicion_categorias[["date"] + cruzadas], on="date", how="left")
    base = base.merge(exposicion_resto.loc[exposicion_resto["family"] == categoria, ["date"] + resto], on="date", how="left")
    assert len(base) == filas and not base.duplicated(["item_nbr", "date"]).any(), "una unión duplicó o perdió filas"

    # Excluir la ventana del terremoto (D-09), los días de programa (D-26, A-03) y las filas con bloques nulos
    base = base.loc[~base["ventana_terremoto"] & ~base["dia_programa"]]
    filas_fase1 = len(base.dropna(subset=[variable_dependiente] + regresores_fase1))
    base = base.dropna(subset=[variable_dependiente] + regresores)
    assert len(base) == filas_fase1, "las variables de otras categorías cambiaron la muestra de la fase 1"

    # Pasar los indicadores a 0/1 para la fórmula
    base[["onpromotion"] + controles_calendario] = base[["onpromotion"] + controles_calendario].astype(int)

    return base, regresores

## Paso 4. Estimar los 10 modelos

`pyfixest` puede avisar que una varianza salió negativa: ocurre en controles de calendario que marcan uno o dos días. El `assert` comprueba que **ninguna variable de promoción** quede sin error estándar.

In [4]:
modelos, bases, regresores = {}, {}, {}
for categoria in categorias:
    bases[categoria], regresores[categoria] = armar_base(categoria)
    formula = (f"{variable_dependiente} ~ {' + '.join(regresores[categoria] + controles_calendario)}"
               f" | {efectos_fijos}")
    modelos[categoria] = pf.fepois(formula, data=bases[categoria], vcov={"CRV1": agrupamiento})

    # Verificar convergencia y errores estándar de las variables de promoción
    promocion = [r for r in regresores[categoria] if not r.startswith("p__RESTO")]
    assert modelos[categoria]._convergence, f"{categoria}: el modelo no convergió"
    assert modelos[categoria].se()[promocion].notna().all(), f"{categoria}: variable de promoción sin error estándar"

# Indicadores de ajuste y tamaño de cada modelo
ajuste = pd.DataFrame(
    {
        "grupo": ["con programa" if c in categorias_programa else "sin programa" for c in categorias],
        "convergio": [modelos[c]._convergence for c in categorias],
        "n_obs": [modelos[c]._N for c in categorias],
        "n_skus": [bases[c]["item_nbr"].nunique() for c in categorias],
        "n_dias": [bases[c]["date"].nunique() for c in categorias],
        "n_semanas": [bases[c]["semana"].nunique() for c in categorias],
        "pseudo_r2": [modelos[c]._pseudo_r2 for c in categorias],
        "deviance": [modelos[c].deviance for c in categorias],
        "valores_propios_negativos_vcov": [(np.linalg.eigvalsh(modelos[c]._vcov) < 0).sum() for c in categorias],
    },
    index=pd.Index(categorias, name="categoria"),
)
ajuste.round(3)

/usr/local/lib/python3.13/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


/usr/local/lib/python3.13/site-packages/pyfixest/estimation/models/fepois_.py:576: UserWarning: 188 observations removed because of separation.
  warnings.warn(


,grupo,convergio,n_obs,n_skus,n_dias,n_semanas,pseudo_r2,deviance,valores_propios_negativos_vcov
categoria,,,,,,,,,
BEVERAGES,sin programa,True,258914,576,532,78,0.721,2661806.106,10
GROCERY I,sin programa,True,556531,1153,532,78,0.555,2634855.466,4
CLEANING,sin programa,True,197454,396,532,78,0.513,740498.645,6
DAIRY,sin programa,True,109779,226,532,78,0.533,390881.919,5
BREAD/BAKERY,sin programa,True,54466,105,532,78,0.670,220396.847,11
"LIQUOR,WINE,BEER",sin programa,True,21466,49,532,78,0.666,105689.791,17
MEATS,con programa,True,24085,55,455,78,0.579,101865.900,12
POULTRY,con programa,True,20977,48,455,78,0.608,178153.803,18
DELI,con programa,True,33486,76,455,78,0.427,87496.685,10


**Qué esperar.** Las mismas observaciones de la fase 1: 532 días en las categorías sin programa y 455–456 en las con programa. MEATS pierde algunas filas por *singletons* o separación. `valores_propios_negativos_vcov` > 0 indica que la matriz agrupada en dos vías no es semidefinida positiva; eso se corrige al calcular los efectos netos (Paso 5).

## Paso 5. Efectos entre categorías: coeficientes, porcentajes y netos

Se reportan solo los coeficientes de las otras categorías ($\beta_{jk}$). Las variables propias, de clase y de resto de la categoría siguen en la ecuación como parte de la especificación, pero sus resultados están en la fase 1. `p__RESTO` es un control y no se interpreta.

- Valor p e intervalo de 95 % con una $t$ de $G-1$ grados de libertad, con $G$ el número de semanas o de SKUs (el menor), igual que en la fase 1.
- Efecto en %: $L_{jk} = 100\,(e^{0{,}1\beta}-1)$, el cambio en las ventas de $j$ cuando 10 pp adicionales del surtido de $k$ están en promoción.
- **Efecto neto** (bloque `neto`): $\beta_0+\beta_{1\text{-}3}+\beta_{4\text{-}6}$, es decir, 10 pp adicionales **sostenidos 7 días** (los bloques son promedios). Su error estándar sale de la matriz de covarianzas corregida para que sea semidefinida positiva (Cameron, Gelbach y Miller, 2011).

In [5]:
def combinacion(modelo, variables: list, grados_libertad: int) -> tuple:
    """
    Esta función estima la suma de uno o más coeficientes con su error estándar, valor p e intervalo de 95 %.

    :param modelo:              un modelo estimado con pf.fepois.
    :param variables:           la lista de coeficientes a sumar (uno solo para un coeficiente individual).
    :param grados_libertad:     los grados de libertad de la t (número de grupos de agrupamiento menos 1).
    :return:                    (beta, ee, p_valor, ic_inf, ic_sup).
    """

    # En sumas, la matriz agrupada en dos vías puede no ser semidefinida positiva y subestimar la varianza: se
    # reemplazan sus valores propios negativos por cero (Cameron, Gelbach y Miller, 2011). Los coeficientes
    # individuales usan la matriz original, como en la fase 1.
    vcov = modelo._vcov
    if len(variables) > 1:
        valores, vectores = np.linalg.eigh((vcov + vcov.T) / 2)
        vcov = vectores @ np.diag(np.clip(valores, 0, None)) @ vectores.T

    pesos = modelo.coef().index.isin(variables).astype(float)
    beta = float(pesos @ modelo.coef().to_numpy())
    ee = float(np.sqrt(pesos @ vcov @ pesos))
    t_critico = stats.t.ppf(0.975, grados_libertad)
    p_valor = 2 * stats.t.sf(abs(beta / ee), grados_libertad)
    return beta, ee, p_valor, beta - t_critico * ee, beta + t_critico * ee


filas = []
for categoria in categorias:
    modelo, base = modelos[categoria], bases[categoria]
    grados_libertad = min(base["item_nbr"].nunique(), base["semana"].nunique()) - 1

    for k in [k for k in categorias if k != categoria]:
        # Cada bloque por separado y el efecto neto (suma de los tres bloques)
        variable = columna_categoria[k]
        combinaciones = {nombre_bloque[b]: [f"{variable}{b}"] for b in bloques}
        combinaciones["neto"] = [f"{variable}{b}" for b in bloques]

        for bloque, sumandos in combinaciones.items():
            beta, ee, p_valor, ic_inf, ic_sup = combinacion(modelo, sumandos, grados_libertad)
            filas.append({
                "categoria_afectada_j": categoria, "categoria_en_promocion_k": k, "bloque": bloque,
                "beta": beta, "ee": ee, "p_valor": p_valor, "ic_inf": ic_inf, "ic_sup": ic_sup,
            })

coeficientes = pd.DataFrame(filas)

# Efecto en porcentaje de +10 pp y su intervalo
coeficientes["efecto_pct"] = 100 * np.expm1(delta_p * coeficientes["beta"])
coeficientes["efecto_ic_inf_pct"] = 100 * np.expm1(delta_p * coeficientes["ic_inf"])
coeficientes["efecto_ic_sup_pct"] = 100 * np.expm1(delta_p * coeficientes["ic_sup"])

## Paso 6. Corrección por comparaciones múltiples (D‑15, D‑17)

Benjamini‑Hochberg al 5 %, en dos familias de pruebas:

1. **Día de la promoción:** los 90 efectos $\beta_{jk,0}$ (`INSTRUCCIONES.md` §7). Es la familia que clasifica los pares.
2. **Efecto neto:** los 90 $L^{\text{net}}_{jk}$.

Los rezagos individuales se reportan con su valor p sin corregir.

- `significativo` usa el p corregido (`p_bh`) cuando existe; es la conclusión que se reporta.
- `estrellas` usa el p **sin** corregir (*** < 0,01; ** < 0,05; * < 0,10). Por eso un par puede tener estrellas y no ser significativo.

In [6]:
for bloque in ["dia", "neto"]:
    filtro = coeficientes["bloque"] == bloque
    coeficientes.loc[filtro, "p_bh"] = multipletests(coeficientes.loc[filtro, "p_valor"], alpha=alfa, method="fdr_bh")[1]

assert (coeficientes["bloque"] == "dia").sum() == len(categorias) * (len(categorias) - 1), "faltan efectos cruzados"

coeficientes["significativo"] = coeficientes["p_bh"].fillna(coeficientes["p_valor"]) < alfa
coeficientes["estrellas"] = pd.cut(coeficientes["p_valor"], [0, 0.01, 0.05, 0.10, 1], labels=["***", "**", "*", ""],
                                   include_lowest=True).astype(str)

# Control del tamaño de los coeficientes (INSTRUCCIONES §10.1: |beta| > 5 sugiere colinealidad o separación)
print(f"Máximo |beta| entre categorías: {coeficientes.loc[coeficientes['bloque'] != 'neto', 'beta'].abs().max():.2f}")

Máximo |beta| entre categorías: 4.22


## Paso 7. Tabla de efectos entre categorías

Una fila por par $(j,k)$: $L_{jk}$ del día y $L^{\text{net}}_{jk}$, con su clasificación según el signo y la significancia corregida (D‑17). La clasificación **del día** es la principal (`metodologia.tex`, `INSTRUCCIONES.md` §7). La neta es complementaria: supone 10 pp adicionales **sostenidos 7 días** y descansa en los bloques de rezago, que son muy colineales en las categorías con campañas de varios días.

`fuera_de_soporte` marca los pares en que +10 pp supera la variación habitual de $P_k$ (`rango_p10_p90_k`: diferencia entre los percentiles 90 y 10 en los días de estimación). En esos casos el efecto a 10 pp es una extrapolación: la metodología solo valida recomendaciones dentro del soporte observado.

In [7]:
# Variación habitual de la intensidad de cada categoría en los días de estimación (fuera de la ventana del terremoto)
dias_estimacion = exposicion_categorias.loc[~calendario["ventana_terremoto"].to_numpy()]
rango_p10_p90 = pd.Series(
    {k: dias_estimacion[columna_categoria[k]].quantile(0.9) - dias_estimacion[columna_categoria[k]].quantile(0.1)
     for k in categorias},
    name="rango_p10_p90",
)

# Una fila por par (j, k) con el efecto del día y el neto
columnas_tabla = ["efecto_pct", "efecto_ic_inf_pct", "efecto_ic_sup_pct", "p_valor", "p_bh", "significativo", "estrellas"]
efectos_cruzados = coeficientes.loc[coeficientes["bloque"].isin(["dia", "neto"])].pivot(
    index=["categoria_afectada_j", "categoria_en_promocion_k"], columns="bloque", values=columnas_tabla
)
efectos_cruzados.columns = [f"{b}_{m}" for m, b in efectos_cruzados.columns]
efectos_cruzados = efectos_cruzados.reset_index()

for bloque in ["dia", "neto"]:
    efectos_cruzados[f"clasificacion_{bloque}"] = np.select(
        [efectos_cruzados[f"{bloque}_significativo"] & (efectos_cruzados[f"{bloque}_efecto_pct"] > 0),
         efectos_cruzados[f"{bloque}_significativo"] & (efectos_cruzados[f"{bloque}_efecto_pct"] < 0)],
        ["sinergia", "canibalización"], "sin efecto",
    )
efectos_cruzados["rango_p10_p90_k"] = efectos_cruzados["categoria_en_promocion_k"].map(rango_p10_p90)
efectos_cruzados["fuera_de_soporte"] = efectos_cruzados["rango_p10_p90_k"] < delta_p

print(pd.crosstab(efectos_cruzados["clasificacion_dia"], efectos_cruzados["clasificacion_neto"], margins=True))
print(f"\nPares con p < 0,05 sin corregir (día): {(efectos_cruzados['dia_p_valor'] < alfa).sum()} de "
      f"{len(efectos_cruzados)}; menor p corregido: {efectos_cruzados['dia_p_bh'].min():.3f}")

# Pares con evidencia: p del día < 0,05 sin corregir, o neto significativo tras la corrección
destacados = efectos_cruzados.loc[(efectos_cruzados["dia_p_valor"] < alfa) | efectos_cruzados["neto_significativo"]]
destacados[["categoria_afectada_j", "categoria_en_promocion_k", "dia_efecto_pct", "dia_p_valor", "dia_p_bh",
            "clasificacion_dia", "neto_efecto_pct", "neto_p_bh", "clasificacion_neto", "fuera_de_soporte"]
           ].sort_values("dia_p_valor").round(3)

clasificacion_neto  canibalización  sin efecto  sinergia  All
clasificacion_dia                                            
sin efecto                       4          83         3   90
All                              4          83         3   90

Pares con p < 0,05 sin corregir (día): 20 de 90; menor p corregido: 0.057


,categoria_afectada_j,categoria_en_promocion_k,dia_efecto_pct,dia_p_valor,dia_p_bh,clasificacion_dia,neto_efecto_pct,neto_p_bh,clasificacion_neto,fuera_de_soporte
60,"LIQUOR,WINE,BEER",MEATS,1.999682,0.000654,0.057228,sin efecto,3.127215,0.520944,sin efecto,True
79,POULTRY,MEATS,2.819979,0.001836,0.057228,sin efecto,0.3351,0.847457,sin efecto,True
59,"LIQUOR,WINE,BEER",GROCERY I,25.831157,0.002319,0.057228,sin efecto,23.608926,0.475792,sin efecto,True
71,MEATS,PRODUCE,-8.044674,0.002724,0.057228,sin efecto,-2.646926,0.795263,sin efecto,True
61,"LIQUOR,WINE,BEER",POULTRY,-10.846071,0.003179,0.057228,sin efecto,5.210383,0.702834,sin efecto,True
44,DELI,PRODUCE,-10.47913,0.004911,0.073662,sin efecto,-10.958167,0.076793,sin efecto,True
38,DELI,CLEANING,-13.158363,0.006575,0.07472,sin efecto,3.550542,0.754587,sin efecto,False
0,BEVERAGES,BREAD/BAKERY,-8.962027,0.006642,0.07472,sin efecto,-6.588844,0.199584,sin efecto,False
26,CLEANING,PRODUCE,-7.800438,0.009025,0.090251,sin efecto,-16.922744,0.036141,canibalización,True
85,PRODUCE,DELI,-3.338221,0.016106,0.144421,sin efecto,-3.840352,0.228388,sin efecto,False


## Paso 8. Exportación

In [9]:
# Tablas en CSV (una por resultado) y un Excel con una hoja por tabla
tablas = {
    "ajuste": ajuste,
    "efectos_cruzados": efectos_cruzados.set_index(["categoria_afectada_j", "categoria_en_promocion_k"]),
    "coeficientes": coeficientes.set_index(["categoria_afectada_j", "categoria_en_promocion_k", "bloque"]),
}
with pd.ExcelWriter(os.path.join(ruta_tablas, "fase2_resultados.xlsx")) as excel:
    for nombre, tabla in tablas.items():
        tabla.to_csv(os.path.join(ruta_tablas, f"fase2_{nombre}.csv"))
        tabla.to_excel(excel, sheet_name=nombre)

print(f"Guardado: {len(tablas)} tablas en {ruta_tablas} (CSV y fase2_resultados.xlsx)")
print(">>>> Proceso finalizado <<<<")

Guardado: 5 tablas en resultados/tablas (CSV y fase2_resultados.xlsx)
>>>> Proceso finalizado <<<<
